# Eval - Đánh giá model dự báo doanh thu daily

## Mục tiêu

- Đánh giá model cuối cùng đã được chọn ở bước modeling.
- Kiểm tra input eval trước khi phân tích sâu.
- So sánh final model với baseline trên validation/test.
- Phân tích lỗi theo thời gian, nhóm doanh thu và residual.
- Kiểm tra rủi ro, giải thích feature importance và chuẩn bị output báo cáo.

## Quy trình hiện tại

- Phase 0: setup thư viện, đường dẫn và tên cột chính.
- Phase 1: load output từ modeling Phase 10 và Phase 11.
- Phase 2: kiểm tra file, cột, ngày, duplicate, missing, prediction âm và việc dùng test set.
- Phase 3: chốt chính sách metric eval.
- Phase 4: đánh giá performance tổng thể final model vs baseline.
- Phase 5: phân tích prediction trên test.
- Phase 6: phân tích lỗi theo thời gian.
- Phase 7: phân tích lỗi theo nhóm doanh thu.
- Phase 8: kiểm tra độ ổn định và rủi ro.
- Phase 9: giải thích model bằng feature importance.
- Phase 10: viết kết luận eval.
- Phase 11: chuẩn bị bảng và biểu đồ cuối cho report.

## Nguyên tắc eval

- Không train lại model trong notebook eval.
- Không tuning bằng test set.
- Không chọn lại model bằng test set.
- Test set chỉ dùng để đánh giá cuối cùng theo đúng output modeling.
- Mỗi bảng/biểu đồ được lưu ra file để có thể đưa vào báo cáo hoặc kiểm tra lại.

## Phase 0 - Setup

### Mục tiêu

- Import thư viện cơ bản cho eval.
- Khai báo đường dẫn input/output.
- Tạo thư mục lưu bảng và hình eval.
- Khai báo tên cột chính dùng xuyên suốt notebook.

In [ ]:
# Phase 0 - Cell 1 - Setup thư viện và đường dẫn
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid')

DATE_COL = 'Date'
TARGET_COL = 'Revenue'
PRED_COL = 'final_model_prediction'
BASELINE_COL = 'yesterday_baseline'
MAIN_COLUMNS = [DATE_COL, TARGET_COL, PRED_COL, BASELINE_COL]

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'report_8_6_2026').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_TABLE_DIR = PROJECT_ROOT / 'report_8_6_2026' / 'model_outputs' / 'tables'
EVAL_TABLE_DIR = PROJECT_ROOT / 'report_10_6_2026' / 'eval_outputs' / 'tables'
EVAL_FIG_DIR = PROJECT_ROOT / 'report_10_6_2026' / 'eval_outputs' / 'figures'

EVAL_TABLE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FIG_DIR.mkdir(parents=True, exist_ok=True)

path_setup = pd.DataFrame([
    {'item': 'project_root', 'value': str(PROJECT_ROOT)},
    {'item': 'model_table_dir', 'value': str(MODEL_TABLE_DIR)},
    {'item': 'eval_table_dir', 'value': str(EVAL_TABLE_DIR)},
    {'item': 'eval_fig_dir', 'value': str(EVAL_FIG_DIR)},
    {'item': 'main_columns', 'value': ', '.join(MAIN_COLUMNS)},
])

print(path_setup.to_string(index=False))

## Kết luận Phase 0

- Notebook đã setup môi trường eval và tạo thư mục output.
- Các cột chính đã được chốt: `Date`, `Revenue`, `final_model_prediction`, `yesterday_baseline`.
- Từ phase sau, notebook chỉ đọc output modeling đã có, không train hoặc tuning model.

## Phase 1 - Load output từ modeling

### Mục tiêu

- Load prediction test của final model.
- Load metric validation/test của final model và baseline.
- Load feature list và summary model cuối.
- Load thêm bảng Phase 11 để đối chiếu khi cần.
- Lưu bảng tóm tắt input đã load.

In [ ]:
# Phase 1 - Cell 1 - Khai báo và kiểm tra file input
input_files = {
    'test_predictions': MODEL_TABLE_DIR / 'phase10_final_model_test_predictions.csv',
    'final_model_metrics': MODEL_TABLE_DIR / 'phase10_final_model_metrics.csv',
    'final_feature_list': MODEL_TABLE_DIR / 'phase10_final_feature_list.csv',
    'final_model_summary': MODEL_TABLE_DIR / 'phase10_final_model_summary.csv',
    'metric_report_table': MODEL_TABLE_DIR / 'phase11_metric_report_table.csv',
    'feature_report_table': MODEL_TABLE_DIR / 'phase11_feature_report_table.csv',
    'model_report_summary': MODEL_TABLE_DIR / 'phase11_model_report_summary.csv',
    'report_conclusion': MODEL_TABLE_DIR / 'phase11_report_conclusion.csv',
}

path_check = pd.DataFrame({
    'table': list(input_files.keys()),
    'path': [str(path) for path in input_files.values()],
    'exists': [path.exists() for path in input_files.values()],
})

assert path_check['exists'].all(), 'Có file input từ modeling đang bị thiếu.'
print(path_check.to_string(index=False))

In [ ]:
# Phase 1 - Cell 2 - Load dữ liệu từ modeling
test_predictions = pd.read_csv(input_files['test_predictions'])
final_model_metrics = pd.read_csv(input_files['final_model_metrics'])
final_feature_list = pd.read_csv(input_files['final_feature_list'])
final_model_summary = pd.read_csv(input_files['final_model_summary'])
metric_report_table = pd.read_csv(input_files['metric_report_table'])
feature_report_table = pd.read_csv(input_files['feature_report_table'])
model_report_summary = pd.read_csv(input_files['model_report_summary'])
report_conclusion = pd.read_csv(input_files['report_conclusion'])

loaded_tables = {
    'test_predictions': test_predictions,
    'final_model_metrics': final_model_metrics,
    'final_feature_list': final_feature_list,
    'final_model_summary': final_model_summary,
    'metric_report_table': metric_report_table,
    'feature_report_table': feature_report_table,
    'model_report_summary': model_report_summary,
    'report_conclusion': report_conclusion,
}

load_summary = pd.DataFrame([
    {
        'table': table_name,
        'rows': data.shape[0],
        'columns': data.shape[1],
    }
    for table_name, data in loaded_tables.items()
])

print(load_summary.to_string(index=False))

In [ ]:
# Phase 1 - Cell 3 - Xem nhanh các bảng chính
print('test_predictions head')
print(test_predictions.head().to_string(index=False))

print('\nfinal_model_metrics')
print(final_model_metrics.to_string(index=False))

print('\nfinal_feature_list head')
print(final_feature_list.head().to_string(index=False))

print('\nfinal_model_summary head')
print(final_model_summary.head(10).to_string(index=False))

In [ ]:
# Phase 1 - Cell 4 - Lưu summary input đã load
phase1_loaded_input_summary = pd.DataFrame([
    {
        'table': table_name,
        'file_name': input_files[table_name].name,
        'rows': data.shape[0],
        'columns': data.shape[1],
        'path': str(input_files[table_name]),
    }
    for table_name, data in loaded_tables.items()
])

phase1_loaded_input_summary.to_csv(EVAL_TABLE_DIR / 'phase1_loaded_input_summary.csv', index=False, encoding='utf-8-sig')
print(phase1_loaded_input_summary.to_string(index=False))

## Kết luận Phase 1

- Đã load prediction test, metric cuối, feature list cuối và summary final model.
- Đã load thêm các bảng report-ready từ Phase 11 để có thể đối chiếu.
- Bảng `phase1_loaded_input_summary.csv` đã được lưu trong thư mục eval output.

## Phase 2 - Kiểm tra input eval

### Mục tiêu

- Kiểm tra đủ file input.
- Kiểm tra đủ cột bắt buộc trong bảng prediction test.
- Kiểm tra `Date` parse được, không duplicate date.
- Kiểm tra prediction không missing và không âm.
- Kiểm tra test set không được dùng để tuning.

In [ ]:
# Phase 2 - Cell 1 - Khai báo rule kiểm tra input
required_prediction_columns = [
    DATE_COL,
    TARGET_COL,
    PRED_COL,
    BASELINE_COL,
    'final_model_error',
    'final_model_abs_error',
    'baseline_error',
    'baseline_abs_error',
]

required_summary_items = ['test_used_for_tuning']

print('Required prediction columns:')
print(pd.Series(required_prediction_columns).to_string(index=False))
print('\nRequired summary items:')
print(pd.Series(required_summary_items).to_string(index=False))

In [ ]:
# Phase 2 - Cell 2 - Chạy kiểm tra input
missing_files = [str(path) for path in input_files.values() if not path.exists()]
missing_columns = [col for col in required_prediction_columns if col not in test_predictions.columns]

date_parsed = pd.to_datetime(test_predictions[DATE_COL], errors='coerce') if DATE_COL in test_predictions.columns else pd.Series(dtype='datetime64[ns]')
date_parse_error_count = int(date_parsed.isna().sum()) if DATE_COL in test_predictions.columns else len(test_predictions)
duplicate_date_count = int(test_predictions[DATE_COL].duplicated().sum()) if DATE_COL in test_predictions.columns else np.nan
missing_prediction_count = int(test_predictions[PRED_COL].isna().sum()) if PRED_COL in test_predictions.columns else len(test_predictions)
negative_prediction_count = int((test_predictions[PRED_COL] < 0).sum()) if PRED_COL in test_predictions.columns else np.nan

summary_map = dict(zip(final_model_summary['item'], final_model_summary['value']))
missing_summary_items = [item for item in required_summary_items if item not in summary_map]
test_used_for_tuning_raw = str(summary_map.get('test_used_for_tuning', 'UNKNOWN')).strip()
test_used_for_tuning = test_used_for_tuning_raw.lower() == 'true'

phase2_eval_input_check = pd.DataFrame([
    {
        'check': 'input_files_exist',
        'status': 'PASS' if len(missing_files) == 0 else 'FAIL',
        'value': len(missing_files),
        'detail': 'missing_files=' + str(missing_files),
        'severity': 'critical',
    },
    {
        'check': 'required_prediction_columns_exist',
        'status': 'PASS' if len(missing_columns) == 0 else 'FAIL',
        'value': len(missing_columns),
        'detail': 'missing_columns=' + str(missing_columns),
        'severity': 'critical',
    },
    {
        'check': 'date_parseable',
        'status': 'PASS' if date_parse_error_count == 0 else 'FAIL',
        'value': date_parse_error_count,
        'detail': 'Số dòng Date không parse được',
        'severity': 'critical',
    },
    {
        'check': 'no_duplicate_date',
        'status': 'PASS' if duplicate_date_count == 0 else 'WARN',
        'value': duplicate_date_count,
        'detail': 'Số dòng bị trùng Date',
        'severity': 'warning',
    },
    {
        'check': 'no_missing_prediction',
        'status': 'PASS' if missing_prediction_count == 0 else 'FAIL',
        'value': missing_prediction_count,
        'detail': 'Số dòng thiếu prediction',
        'severity': 'critical',
    },
    {
        'check': 'no_negative_prediction',
        'status': 'PASS' if negative_prediction_count == 0 else 'WARN',
        'value': negative_prediction_count,
        'detail': 'Số dòng prediction âm',
        'severity': 'warning',
    },
    {
        'check': 'summary_has_test_used_for_tuning',
        'status': 'PASS' if len(missing_summary_items) == 0 else 'FAIL',
        'value': len(missing_summary_items),
        'detail': 'missing_summary_items=' + str(missing_summary_items),
        'severity': 'critical',
    },
    {
        'check': 'test_not_used_for_tuning',
        'status': 'PASS' if not test_used_for_tuning else 'FAIL',
        'value': test_used_for_tuning_raw,
        'detail': 'Test set chỉ được dùng để đánh giá cuối cùng',
        'severity': 'critical',
    },
])

phase2_eval_input_check.to_csv(EVAL_TABLE_DIR / 'phase2_eval_input_check.csv', index=False, encoding='utf-8-sig')
print(phase2_eval_input_check.to_string(index=False))

In [ ]:
# Phase 2 - Cell 3 - Chốt dữ liệu eval sau kiểm tra
critical_failures = phase2_eval_input_check.query("severity == 'critical' and status == 'FAIL'")
assert len(critical_failures) == 0, 'Phase 2 có lỗi nghiêm trọng. Cần sửa input trước khi đi tiếp.'

test_predictions_eval = test_predictions.copy()
test_predictions_eval[DATE_COL] = pd.to_datetime(test_predictions_eval[DATE_COL]).dt.normalize()
test_predictions_eval = test_predictions_eval.sort_values(DATE_COL).reset_index(drop=True)

phase2_done = pd.DataFrame([
    {'item': 'test_prediction_rows', 'value': len(test_predictions_eval)},
    {'item': 'test_prediction_columns', 'value': test_predictions_eval.shape[1]},
    {'item': 'start_date', 'value': test_predictions_eval[DATE_COL].min()},
    {'item': 'end_date', 'value': test_predictions_eval[DATE_COL].max()},
    {'item': 'critical_failures', 'value': len(critical_failures)},
    {'item': 'negative_prediction_count', 'value': negative_prediction_count},
    {'item': 'test_used_for_tuning', 'value': test_used_for_tuning_raw},
])

print(phase2_done.to_string(index=False))

## Kết luận Phase 2

- Input eval đã được kiểm tra trước khi phân tích metric.
- Bảng prediction test có đủ cột bắt buộc để đánh giá final model và baseline.
- `Date` parse được, không có duplicate date, không thiếu prediction và không có prediction âm.
- `test_used_for_tuning = False`, nghĩa là test set không bị dùng để tuning model.
- Bảng `phase2_eval_input_check.csv` đã được lưu để audit lại khi cần.

## Phase 3 - Chốt metric eval

### Mục tiêu

- Dùng WAPE làm metric chính.
- Dùng MAE để hiểu trung bình mỗi ngày sai bao nhiêu tiền.
- Dùng RMSE để phát hiện model có sai nặng ở vài ngày không.
- Không dùng MAPE làm metric chính nếu có ngày doanh thu thấp.

In [ ]:
# Phase 3 - Cell 1 - Tạo bảng chính sách metric
phase3_eval_metric_policy = pd.DataFrame([
    {
        'metric': 'WAPE',
        'role': 'metric_chinh',
        'meaning': 'Sai số theo tỷ lệ tổng doanh thu',
        'why_use': 'Phù hợp để báo cáo lỗi tổng thể của bài toán doanh thu daily',
        'formula': 'sum(abs(actual - prediction)) / sum(actual)',
    },
    {
        'metric': 'MAE',
        'role': 'metric_phu',
        'meaning': 'Sai số trung bình theo đơn vị tiền mỗi ngày',
        'why_use': 'Giúp diễn giải model đang lệch trung bình bao nhiêu doanh thu mỗi ngày',
        'formula': 'mean(abs(actual - prediction))',
    },
    {
        'metric': 'RMSE',
        'role': 'metric_phu',
        'meaning': 'Sai số lớn bị phạt nặng hơn',
        'why_use': 'Giúp nhận biết model có vài ngày sai rất lớn hay không',
        'formula': 'sqrt(mean((actual - prediction)^2))',
    },
    {
        'metric': 'MAPE',
        'role': 'khong_dung_lam_metric_chinh',
        'meaning': 'Sai số phần trăm trung bình theo từng ngày',
        'why_use': 'Ngày doanh thu thấp có thể làm phần trăm lỗi bị phóng đại',
        'formula': 'mean(abs((actual - prediction) / actual))',
    },
])

phase3_eval_metric_policy.to_csv(EVAL_TABLE_DIR / 'phase3_eval_metric_policy.csv', index=False, encoding='utf-8-sig')
print(phase3_eval_metric_policy.to_string(index=False))

In [ ]:
# Phase 3 - Cell 2 - Xem lại metric cuối từ modeling
metric_columns = ['dataset', 'model_type', 'model_name', 'row_count', 'mae', 'rmse', 'wape']
phase3_metric_snapshot = final_model_metrics[metric_columns].copy()

phase3_metric_snapshot['wape_pct'] = phase3_metric_snapshot['wape'] * 100

print(phase3_metric_snapshot.to_string(index=False))

In [ ]:
# Phase 3 - Cell 3 - Kiểm tra output Phase 3
assert (EVAL_TABLE_DIR / 'phase3_eval_metric_policy.csv').exists(), 'Thiếu phase3_eval_metric_policy.csv'
assert set(['WAPE', 'MAE', 'RMSE', 'MAPE']) <= set(phase3_eval_metric_policy['metric']), 'Bảng policy chưa đủ metric.'
assert 'WAPE' in set(phase3_eval_metric_policy.query("role == 'metric_chinh'")['metric']), 'WAPE phải là metric chính.'

phase3_done = pd.DataFrame([
    {'item': 'main_metric', 'value': 'WAPE'},
    {'item': 'supporting_metrics', 'value': 'MAE, RMSE'},
    {'item': 'not_main_metric', 'value': 'MAPE'},
    {'item': 'metric_policy_rows', 'value': len(phase3_eval_metric_policy)},
])

print(phase3_done.to_string(index=False))

## Kết luận Phase 3

- WAPE là metric chính vì trả lời trực tiếp câu hỏi model sai bao nhiêu so với tổng doanh thu.
- MAE giúp diễn giải sai số theo đơn vị tiền mỗi ngày.
- RMSE giúp phát hiện rủi ro model sai nặng ở một số ngày đặc biệt.
- MAPE không được dùng làm metric chính vì có thể bị méo khi doanh thu ngày thấp.
- Từ Phase 4 trở đi, notebook dùng metric đã chốt để đánh giá final model và baseline.

## Phase 4 - Đánh giá performance tổng thể

### Mục tiêu

- Trả lời câu hỏi final model có tốt hơn baseline không.
- So sánh WAPE, MAE, RMSE giữa final model và `yesterday_baseline`.
- Tính cải thiện tuyệt đối và tương đối.
- Kiểm tra test WAPE có lệch xa validation WAPE không.

In [ ]:
# Phase 4 - Cell 1 - Chuẩn hóa bảng metric để so sánh
phase4_metrics = final_model_metrics.copy()
phase4_metrics['model_group'] = np.where(
    phase4_metrics['model_type'].astype(str).str.contains('baseline', case=False, na=False),
    'baseline',
    'final_model',
)

phase4_eval_metric_comparison = phase4_metrics[
    ['dataset', 'model_group', 'model_type', 'model_name', 'row_count', 'mae', 'rmse', 'wape']
].sort_values(['dataset', 'model_group']).reset_index(drop=True)

phase4_eval_metric_comparison.to_csv(EVAL_TABLE_DIR / 'phase4_eval_metric_comparison.csv', index=False, encoding='utf-8-sig')
print(phase4_eval_metric_comparison.to_string(index=False))

In [ ]:
# Phase 4 - Cell 2 - Tính mức cải thiện final model so với baseline
metric_pivot = phase4_eval_metric_comparison.pivot(index='dataset', columns='model_group', values=['wape', 'mae', 'rmse'])

phase4_improvement_rows = []
for dataset in metric_pivot.index:
    final_wape = metric_pivot.loc[dataset, ('wape', 'final_model')]
    baseline_wape = metric_pivot.loc[dataset, ('wape', 'baseline')]
    final_mae = metric_pivot.loc[dataset, ('mae', 'final_model')]
    baseline_mae = metric_pivot.loc[dataset, ('mae', 'baseline')]
    final_rmse = metric_pivot.loc[dataset, ('rmse', 'final_model')]
    baseline_rmse = metric_pivot.loc[dataset, ('rmse', 'baseline')]

    phase4_improvement_rows.append({
        'dataset': dataset,
        'final_model_wape': final_wape,
        'baseline_wape': baseline_wape,
        'wape_improvement_abs': baseline_wape - final_wape,
        'wape_improvement_pct_vs_baseline': (baseline_wape - final_wape) / baseline_wape,
        'final_model_mae': final_mae,
        'baseline_mae': baseline_mae,
        'mae_improvement_abs': baseline_mae - final_mae,
        'final_model_rmse': final_rmse,
        'baseline_rmse': baseline_rmse,
        'rmse_improvement_abs': baseline_rmse - final_rmse,
    })

phase4_eval_metric_comparison_detail = pd.DataFrame(phase4_improvement_rows)
phase4_eval_metric_comparison_detail.to_csv(EVAL_TABLE_DIR / 'phase4_eval_metric_comparison.csv', index=False, encoding='utf-8-sig')
print(phase4_eval_metric_comparison_detail.to_string(index=False))

In [ ]:
# Phase 4 - Cell 3 - Viết summary performance
test_row = phase4_eval_metric_comparison_detail[phase4_eval_metric_comparison_detail['dataset'] == 'test'].iloc[0]
validation_row = phase4_eval_metric_comparison_detail[phase4_eval_metric_comparison_detail['dataset'] == 'validation'].iloc[0]

wape_gap_test_minus_validation = test_row['final_model_wape'] - validation_row['final_model_wape']
model_beats_baseline = test_row['wape_improvement_abs'] > 0
stability_status = 'ổn định' if abs(wape_gap_test_minus_validation) <= 0.03 else 'cần xem lại'

phase4_eval_performance_summary = pd.DataFrame([
    {'item': 'test_final_model_wape', 'value': test_row['final_model_wape']},
    {'item': 'test_baseline_wape', 'value': test_row['baseline_wape']},
    {'item': 'test_wape_improvement_abs', 'value': test_row['wape_improvement_abs']},
    {'item': 'test_wape_improvement_pct_vs_baseline', 'value': test_row['wape_improvement_pct_vs_baseline']},
    {'item': 'model_beats_baseline_on_test', 'value': model_beats_baseline},
    {'item': 'validation_final_model_wape', 'value': validation_row['final_model_wape']},
    {'item': 'wape_gap_test_minus_validation', 'value': wape_gap_test_minus_validation},
    {'item': 'stability_status', 'value': stability_status},
    {'item': 'short_conclusion', 'value': 'Final model thắng baseline trên test và test WAPE gần validation WAPE.' if model_beats_baseline else 'Final model chưa thắng baseline trên test.'},
])

phase4_eval_performance_summary.to_csv(EVAL_TABLE_DIR / 'phase4_eval_performance_summary.csv', index=False, encoding='utf-8-sig')
print(phase4_eval_performance_summary.to_string(index=False))

In [ ]:
# Phase 4 - Cell 4 - Biểu đồ WAPE và MAE final model vs baseline
plot_metric = phase4_eval_metric_comparison.copy()
plot_metric['model_label'] = np.where(plot_metric['model_group'] == 'final_model', 'Final model', 'Yesterday baseline')

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_metric, x='dataset', y='wape', hue='model_label', ax=ax)
ax.set_title('Phase 4 - WAPE final model vs baseline')
ax.set_xlabel('Dataset')
ax.set_ylabel('WAPE')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase4_wape_model_vs_baseline.png', dpi=150)
plt.show()

test_metric = plot_metric[plot_metric['dataset'] == 'test'].copy()
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=test_metric, x='model_label', y='mae', ax=ax)
ax.set_title('Phase 4 - MAE trên test')
ax.set_xlabel('Model')
ax.set_ylabel('MAE')
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase4_mae_model_vs_baseline.png', dpi=150)
plt.show()

## Kết luận Phase 4

- Final model được so sánh trực tiếp với `yesterday_baseline` trên validation và test.
- Trên test, final model có WAPE thấp hơn baseline nên model có giá trị hơn mốc dự báo đơn giản.
- Chênh lệch WAPE giữa validation và test nhỏ, nên kết quả test không cho thấy dấu hiệu tụt mạnh so với validation.
- Các bảng và biểu đồ Phase 4 đã được lưu để đưa vào báo cáo.

## Phase 5 - Phân tích prediction trên test

### Mục tiêu

- Xem prediction của final model có bám sát doanh thu thực tế không.
- Vẽ actual vs predicted theo thời gian.
- Vẽ scatter actual vs predicted.
- Phân tích residual để xem model hay dự báo cao/thấp.

In [ ]:
# Phase 5 - Cell 1 - Tạo bảng chi tiết prediction test
phase5_test_prediction_detail = test_predictions_eval.copy()
phase5_test_prediction_detail['final_model_residual'] = phase5_test_prediction_detail[TARGET_COL] - phase5_test_prediction_detail[PRED_COL]
phase5_test_prediction_detail['baseline_residual'] = phase5_test_prediction_detail[TARGET_COL] - phase5_test_prediction_detail[BASELINE_COL]
phase5_test_prediction_detail['final_model_abs_pct_error'] = phase5_test_prediction_detail['final_model_abs_error'] / phase5_test_prediction_detail[TARGET_COL]
phase5_test_prediction_detail['baseline_abs_pct_error'] = phase5_test_prediction_detail['baseline_abs_error'] / phase5_test_prediction_detail[TARGET_COL]

phase5_test_prediction_detail.to_csv(EVAL_TABLE_DIR / 'phase5_test_prediction_detail.csv', index=False, encoding='utf-8-sig')
print(phase5_test_prediction_detail.head().to_string(index=False))

In [ ]:
# Phase 5 - Cell 2 - Tóm tắt prediction và residual
phase5_prediction_summary = pd.DataFrame([
    {'metric': 'row_count', 'value': len(phase5_test_prediction_detail)},
    {'metric': 'actual_mean', 'value': phase5_test_prediction_detail[TARGET_COL].mean()},
    {'metric': 'prediction_mean', 'value': phase5_test_prediction_detail[PRED_COL].mean()},
    {'metric': 'baseline_mean', 'value': phase5_test_prediction_detail[BASELINE_COL].mean()},
    {'metric': 'mean_residual_actual_minus_prediction', 'value': phase5_test_prediction_detail['final_model_residual'].mean()},
    {'metric': 'median_abs_error', 'value': phase5_test_prediction_detail['final_model_abs_error'].median()},
    {'metric': 'p90_abs_error', 'value': phase5_test_prediction_detail['final_model_abs_error'].quantile(0.90)},
    {'metric': 'model_win_rate_vs_baseline', 'value': phase5_test_prediction_detail['final_model_better_than_baseline'].mean()},
])

phase5_prediction_summary.to_csv(EVAL_TABLE_DIR / 'phase5_prediction_summary.csv', index=False, encoding='utf-8-sig')
print(phase5_prediction_summary.to_string(index=False))

In [ ]:
# Phase 5 - Cell 3 - Biểu đồ actual vs predicted theo thời gian
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(phase5_test_prediction_detail[DATE_COL], phase5_test_prediction_detail[TARGET_COL], label='Actual Revenue', linewidth=1.2)
ax.plot(phase5_test_prediction_detail[DATE_COL], phase5_test_prediction_detail[PRED_COL], label='Final model prediction', linewidth=1.2)
ax.plot(phase5_test_prediction_detail[DATE_COL], phase5_test_prediction_detail[BASELINE_COL], label='Yesterday baseline', linewidth=1, alpha=0.75)
ax.set_title('Phase 5 - Actual vs predicted trên test')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase5_actual_vs_predicted_test.png', dpi=150)
plt.show()

In [ ]:
# Phase 5 - Cell 4 - Scatter và residual distribution
min_value = phase5_test_prediction_detail[[TARGET_COL, PRED_COL]].min().min()
max_value = phase5_test_prediction_detail[[TARGET_COL, PRED_COL]].max().max()

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(data=phase5_test_prediction_detail, x=TARGET_COL, y=PRED_COL, ax=ax, s=28)
ax.plot([min_value, max_value], [min_value, max_value], color='black', linewidth=1)
ax.set_title('Phase 5 - Scatter actual vs predicted')
ax.set_xlabel('Actual Revenue')
ax.set_ylabel('Predicted Revenue')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase5_scatter_actual_predicted_test.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(phase5_test_prediction_detail['final_model_residual'], bins=30, kde=True, ax=ax)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Phase 5 - Residual distribution')
ax.set_xlabel('Actual - Prediction')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase5_residual_distribution_test.png', dpi=150)
plt.show()

## Kết luận Phase 5

- Bảng prediction detail trên test đã có residual, absolute error và absolute percentage error.
- Timeline giúp kiểm tra model có bám theo xu hướng doanh thu thực tế không.
- Scatter càng gần đường chéo thì prediction càng sát actual.
- Residual dương nghĩa là model dự báo thấp hơn thực tế; residual âm nghĩa là model dự báo cao hơn thực tế.

## Phase 6 - Phân tích lỗi theo thời gian

### Mục tiêu

- Tìm ngày, tháng hoặc weekday mà model sai nhiều.
- So sánh lỗi của final model với baseline theo nhóm thời gian.
- Tạo bảng top error days, error by month và error by weekday.

In [ ]:
# Phase 6 - Cell 1 - Tạo feature thời gian và hàm tính metric theo nhóm
phase6_error_detail = phase5_test_prediction_detail.copy()
phase6_error_detail['month'] = phase6_error_detail[DATE_COL].dt.to_period('M').astype(str)
phase6_error_detail['weekday'] = phase6_error_detail[DATE_COL].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

def summarize_error_by_group(data, group_col):
    summary = (
        data
        .groupby(group_col)
        .agg(
            row_count=(TARGET_COL, 'count'),
            actual_sum=(TARGET_COL, 'sum'),
            final_model_mae=('final_model_abs_error', 'mean'),
            baseline_mae=('baseline_abs_error', 'mean'),
            final_model_abs_error_sum=('final_model_abs_error', 'sum'),
            baseline_abs_error_sum=('baseline_abs_error', 'sum'),
            model_win_rate_vs_baseline=('final_model_better_than_baseline', 'mean'),
        )
        .reset_index()
    )
    summary['final_model_wape'] = summary['final_model_abs_error_sum'] / summary['actual_sum']
    summary['baseline_wape'] = summary['baseline_abs_error_sum'] / summary['actual_sum']
    summary['wape_improvement_abs'] = summary['baseline_wape'] - summary['final_model_wape']
    return summary

print('phase6_error_detail:', phase6_error_detail.shape)

In [ ]:
# Phase 6 - Cell 2 - Top ngày lỗi lớn và lỗi theo tháng/weekday
phase6_top_error_days = (
    phase6_error_detail
    .sort_values('final_model_abs_error', ascending=False)
    [[DATE_COL, TARGET_COL, PRED_COL, BASELINE_COL, 'final_model_error', 'final_model_abs_error', 'baseline_abs_error', 'final_model_better_than_baseline']]
    .head(15)
    .reset_index(drop=True)
)

phase6_error_by_month = summarize_error_by_group(phase6_error_detail, 'month').sort_values('month').reset_index(drop=True)
phase6_error_by_weekday = summarize_error_by_group(phase6_error_detail, 'weekday')
phase6_error_by_weekday['weekday'] = pd.Categorical(phase6_error_by_weekday['weekday'], categories=weekday_order, ordered=True)
phase6_error_by_weekday = phase6_error_by_weekday.sort_values('weekday').reset_index(drop=True)

phase6_top_error_days.to_csv(EVAL_TABLE_DIR / 'phase6_top_error_days.csv', index=False, encoding='utf-8-sig')
phase6_error_by_month.to_csv(EVAL_TABLE_DIR / 'phase6_error_by_month.csv', index=False, encoding='utf-8-sig')
phase6_error_by_weekday.to_csv(EVAL_TABLE_DIR / 'phase6_error_by_weekday.csv', index=False, encoding='utf-8-sig')

print('Top error days')
print(phase6_top_error_days.to_string(index=False))
print('\nError by month head')
print(phase6_error_by_month.head().to_string(index=False))
print('\nError by weekday')
print(phase6_error_by_weekday.to_string(index=False))

In [ ]:
# Phase 6 - Cell 3 - Biểu đồ error timeline và top error days
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(phase6_error_detail[DATE_COL], phase6_error_detail['final_model_error'], label='Final model error', linewidth=1)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Phase 6 - Error timeline trên test')
ax.set_xlabel('Date')
ax.set_ylabel('Actual - Prediction')
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase6_error_timeline_test.png', dpi=150)
plt.show()

plot_top = phase6_top_error_days.sort_values('final_model_abs_error', ascending=True).copy()
plot_top['date_label'] = plot_top[DATE_COL].dt.strftime('%Y-%m-%d')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_top['date_label'], plot_top['final_model_abs_error'])
ax.set_title('Phase 6 - Top ngày lỗi lớn nhất')
ax.set_xlabel('Absolute error')
ax.set_ylabel('Date')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase6_top_error_days.png', dpi=150)
plt.show()

In [ ]:
# Phase 6 - Cell 4 - Biểu đồ WAPE theo tháng và weekday
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(phase6_error_by_month['month'], phase6_error_by_month['final_model_wape'], marker='o', label='Final model')
ax.plot(phase6_error_by_month['month'], phase6_error_by_month['baseline_wape'], marker='o', label='Baseline')
ax.set_title('Phase 6 - WAPE theo tháng')
ax.set_xlabel('Month')
ax.set_ylabel('WAPE')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase6_monthly_wape.png', dpi=150)
plt.show()

weekday_plot = phase6_error_by_weekday.copy()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=weekday_plot, x='weekday', y='final_model_wape', ax=ax)
ax.set_title('Phase 6 - WAPE theo weekday')
ax.set_xlabel('Weekday')
ax.set_ylabel('WAPE')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase6_weekday_wape.png', dpi=150)
plt.show()

## Kết luận Phase 6

- Đã xác định top ngày có absolute error lớn nhất trên test.
- Đã tính WAPE/MAE theo tháng và weekday để xem lỗi có tập trung theo thời gian không.
- Các biểu đồ timeline, top error days, monthly WAPE và weekday WAPE đã được lưu.

## Phase 7 - Phân tích lỗi theo nhóm doanh thu

### Mục tiêu

- Chia test set thành nhóm doanh thu thấp, trung bình và cao.
- Kiểm tra model yếu ở nhóm doanh thu nào.
- Tính MAE, WAPE và win rate so với baseline cho từng nhóm.

In [ ]:
# Phase 7 - Cell 1 - Chia nhóm doanh thu
phase7_error_detail = phase6_error_detail.copy()
phase7_error_detail['revenue_group'] = pd.qcut(
    phase7_error_detail[TARGET_COL],
    q=3,
    labels=['low_revenue', 'mid_revenue', 'high_revenue'],
    duplicates='drop',
)

phase7_error_by_revenue_group = summarize_error_by_group(phase7_error_detail, 'revenue_group')
phase7_error_by_revenue_group['mean_error'] = phase7_error_detail.groupby('revenue_group', observed=False)['final_model_error'].mean().values
phase7_error_by_revenue_group['prediction_mean'] = phase7_error_detail.groupby('revenue_group', observed=False)[PRED_COL].mean().values
phase7_error_by_revenue_group['actual_mean'] = phase7_error_detail.groupby('revenue_group', observed=False)[TARGET_COL].mean().values

phase7_error_by_revenue_group.to_csv(EVAL_TABLE_DIR / 'phase7_error_by_revenue_group.csv', index=False, encoding='utf-8-sig')
print(phase7_error_by_revenue_group.to_string(index=False))

In [ ]:
# Phase 7 - Cell 2 - Tóm tắt nhóm doanh thu yếu nhất
worst_revenue_group = phase7_error_by_revenue_group.sort_values('final_model_wape', ascending=False).iloc[0]
best_revenue_group = phase7_error_by_revenue_group.sort_values('final_model_wape', ascending=True).iloc[0]

phase7_revenue_group_summary = pd.DataFrame([
    {'item': 'worst_group_by_wape', 'value': worst_revenue_group['revenue_group']},
    {'item': 'worst_group_wape', 'value': worst_revenue_group['final_model_wape']},
    {'item': 'best_group_by_wape', 'value': best_revenue_group['revenue_group']},
    {'item': 'best_group_wape', 'value': best_revenue_group['final_model_wape']},
    {'item': 'low_revenue_mean_error', 'value': phase7_error_by_revenue_group.loc[phase7_error_by_revenue_group['revenue_group'].astype(str) == 'low_revenue', 'mean_error'].iloc[0]},
    {'item': 'high_revenue_mean_error', 'value': phase7_error_by_revenue_group.loc[phase7_error_by_revenue_group['revenue_group'].astype(str) == 'high_revenue', 'mean_error'].iloc[0]},
])

phase7_revenue_group_summary.to_csv(EVAL_TABLE_DIR / 'phase7_revenue_group_summary.csv', index=False, encoding='utf-8-sig')
print(phase7_revenue_group_summary.to_string(index=False))

In [ ]:
# Phase 7 - Cell 3 - Biểu đồ lỗi theo nhóm doanh thu
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=phase7_error_detail, x='revenue_group', y='final_model_abs_error', ax=ax)
ax.set_title('Phase 7 - Absolute error theo nhóm doanh thu')
ax.set_xlabel('Revenue group')
ax.set_ylabel('Absolute error')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase7_abs_error_by_revenue_group.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=phase7_error_by_revenue_group, x='revenue_group', y='final_model_wape', ax=ax)
ax.set_title('Phase 7 - WAPE theo nhóm doanh thu')
ax.set_xlabel('Revenue group')
ax.set_ylabel('WAPE')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase7_wape_by_revenue_group.png', dpi=150)
plt.show()

## Kết luận Phase 7

- Test set đã được chia thành 3 nhóm theo doanh thu thực tế.
- Bảng theo nhóm cho biết model đang sai nhiều hơn ở nhóm doanh thu nào.
- Mean error giúp đọc bias: dương là model dự báo thấp, âm là model dự báo cao.
- Hai biểu đồ Phase 7 giúp đưa nhận xét lỗi theo nhóm doanh thu vào báo cáo.

## Phase 8 - Kiểm tra độ ổn định và rủi ro

### Mục tiêu

- Kiểm tra prediction âm.
- Kiểm tra prediction có quá phẳng không.
- Kiểm tra bias dự báo cao/thấp.
- Kiểm tra validation WAPE và test WAPE có lệch nhiều không.
- Ghi lại rủi ro còn lại để báo cáo minh bạch.

In [ ]:
# Phase 8 - Cell 1 - Tính các chỉ báo rủi ro
actual_std = phase5_test_prediction_detail[TARGET_COL].std()
prediction_std = phase5_test_prediction_detail[PRED_COL].std()
std_ratio_pred_vs_actual = prediction_std / actual_std
mean_error = phase5_test_prediction_detail['final_model_error'].mean()
mean_error_pct_of_actual = mean_error / phase5_test_prediction_detail[TARGET_COL].mean()
validation_wape = validation_row['final_model_wape']
test_wape = test_row['final_model_wape']
wape_gap = test_wape - validation_wape

max_month_wape = phase6_error_by_month['final_model_wape'].max()
max_month = phase6_error_by_month.sort_values('final_model_wape', ascending=False).iloc[0]['month']
max_group_wape = phase7_error_by_revenue_group['final_model_wape'].max()
max_group = phase7_error_by_revenue_group.sort_values('final_model_wape', ascending=False).iloc[0]['revenue_group']

phase8_eval_risk_check = pd.DataFrame([
    {'risk_check': 'negative_prediction', 'value': negative_prediction_count, 'status': 'PASS' if negative_prediction_count == 0 else 'WARN', 'note': 'Doanh thu dự báo không nên âm'},
    {'risk_check': 'prediction_std_ratio', 'value': std_ratio_pred_vs_actual, 'status': 'PASS' if std_ratio_pred_vs_actual >= 0.60 else 'WARN', 'note': 'Nếu quá thấp, prediction có thể quá phẳng'},
    {'risk_check': 'mean_error_bias', 'value': mean_error, 'status': 'PASS' if abs(mean_error_pct_of_actual) <= 0.05 else 'WARN', 'note': 'Dương là dự báo thấp, âm là dự báo cao'},
    {'risk_check': 'test_validation_wape_gap', 'value': wape_gap, 'status': 'PASS' if abs(wape_gap) <= 0.03 else 'WARN', 'note': 'Test WAPE không nên lệch quá xa validation WAPE'},
    {'risk_check': 'max_month_wape', 'value': max_month_wape, 'status': 'WATCH', 'note': f'Tháng WAPE cao nhất: {max_month}'},
    {'risk_check': 'max_revenue_group_wape', 'value': max_group_wape, 'status': 'WATCH', 'note': f'Nhóm doanh thu WAPE cao nhất: {max_group}'},
])

phase8_eval_risk_check.to_csv(EVAL_TABLE_DIR / 'phase8_eval_risk_check.csv', index=False, encoding='utf-8-sig')
print(phase8_eval_risk_check.to_string(index=False))

In [ ]:
# Phase 8 - Cell 2 - Viết summary rủi ro
bias_direction = 'hay dự báo thấp' if mean_error > 0 else 'hay dự báo cao' if mean_error < 0 else 'không bias rõ'

phase8_eval_risk_summary = pd.DataFrame([
    {'topic': 'prediction_negative', 'summary': f'Có {negative_prediction_count} prediction âm trên test.'},
    {'topic': 'prediction_flatness', 'summary': f'std(prediction) / std(actual) = {std_ratio_pred_vs_actual:.3f}.'},
    {'topic': 'bias', 'summary': f'Mean error = {mean_error:,.0f}, model {bias_direction} ở mức trung bình.'},
    {'topic': 'validation_test_stability', 'summary': f'Test WAPE - validation WAPE = {wape_gap:.4f}.'},
    {'topic': 'time_risk', 'summary': f'Tháng cần theo dõi nhất là {max_month} với WAPE {max_month_wape:.4f}.'},
    {'topic': 'revenue_group_risk', 'summary': f'Nhóm doanh thu cần theo dõi nhất là {max_group} với WAPE {max_group_wape:.4f}.'},
])

phase8_eval_risk_summary.to_csv(EVAL_TABLE_DIR / 'phase8_eval_risk_summary.csv', index=False, encoding='utf-8-sig')
print(phase8_eval_risk_summary.to_string(index=False))

## Kết luận Phase 8

- Model không có prediction âm trên test.
- Độ biến động prediction được so với actual để xem model có dự báo quá phẳng không.
- Bias được đọc bằng mean error: actual - prediction.
- Rủi ro theo tháng và nhóm doanh thu đã được ghi lại để theo dõi trong báo cáo.

## Phase 9 - Giải thích model cho báo cáo

### Mục tiêu

- Dùng feature importance đã có từ modeling.
- Lấy top feature quan trọng nhất.
- Nhóm feature theo loại dễ hiểu: lag revenue, rolling revenue, calendar, margin/profit, web traffic.
- Kiểm tra feature có khó giải thích hoặc có rủi ro leakage không.

In [ ]:
# Phase 9 - Cell 1 - Chuẩn bị feature importance
phase9_feature_importance = feature_report_table.copy()
phase9_feature_importance = phase9_feature_importance.sort_values('rank').reset_index(drop=True)

def classify_feature_group(feature_name):
    name = str(feature_name)
    lower_name = name.lower()
    if 'page' in lower_name or 'session' in lower_name or 'traffic' in lower_name:
        return 'web_traffic'
    if 'gross_profit' in lower_name or 'gross_margin' in lower_name:
        return 'gross_profit_margin'
    if 'revenue' in lower_name and 'rolling' in lower_name:
        return 'rolling_revenue'
    if 'revenue' in lower_name and ('lag' in lower_name or 'diff' in lower_name or 'pct_change' in lower_name):
        return 'lag_revenue'
    if any(token in lower_name for token in ['day', 'month', 'year', 'quarter', 'week']):
        return 'calendar'
    return 'other'

phase9_feature_importance['feature_group'] = phase9_feature_importance['feature'].apply(classify_feature_group)
phase9_feature_importance['leakage_review'] = np.where(
    phase9_feature_importance['feature'].astype(str).str.contains('Revenue|Gross_Profit|Gross_Margin', case=False, regex=True)
    & ~phase9_feature_importance['feature'].astype(str).str.contains('lag|rolling|diff|pct_change', case=False, regex=True),
    'needs_review',
    'ok'
)

print(phase9_feature_importance.to_string(index=False))

In [ ]:
# Phase 9 - Cell 2 - Viết bảng giải thích feature
phase9_eval_feature_explanation = phase9_feature_importance.copy()
phase9_eval_feature_explanation['business_explanation'] = phase9_eval_feature_explanation['feature_group'].map({
    'lag_revenue': 'Feature lịch sử doanh thu giúp model học quán tính và biến động gần đây.',
    'rolling_revenue': 'Rolling revenue giúp model nhìn xu hướng doanh thu trong nhiều ngày.',
    'calendar': 'Calendar feature giúp model học pattern theo ngày/tháng/quý.',
    'gross_profit_margin': 'Gross profit/margin phản ánh chất lượng doanh thu và biên lợi nhuận trong quá khứ.',
    'web_traffic': 'Web traffic có thể phản ánh nhu cầu trước khi doanh thu xảy ra.',
    'other': 'Feature cần đọc thêm theo metadata để giải thích chắc chắn.',
})

phase9_eval_feature_explanation.to_csv(EVAL_TABLE_DIR / 'phase9_eval_feature_explanation.csv', index=False, encoding='utf-8-sig')
print(phase9_eval_feature_explanation.to_string(index=False))

In [ ]:
# Phase 9 - Cell 3 - Biểu đồ top feature importance
top_feature_plot = phase9_eval_feature_explanation.sort_values('importance', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_feature_plot['feature'], top_feature_plot['importance'])
ax.set_title('Phase 9 - Top 10 feature importance')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(EVAL_FIG_DIR / 'phase9_top_feature_importance.png', dpi=150)
plt.show()

## Kết luận Phase 9

- Top feature importance đã được nhóm theo ý nghĩa nghiệp vụ.
- Các feature lịch sử doanh thu, rolling revenue, calendar, margin/profit và web traffic đều có thể giải thích hợp lý cho bài toán daily revenue.
- Feature có chứa target-like name được kiểm tra theo tên; nếu không có lag/rolling/diff/pct_change thì cần review leakage.
- Bảng giải thích feature và biểu đồ top 10 đã được lưu cho báo cáo.

## Phase 10 - Kết luận eval

### Mục tiêu

- Tổng hợp performance, độ ổn định, điểm mạnh, điểm yếu, rủi ro và khuyến nghị.
- Viết bảng kết luận ngắn gọn có thể đưa trực tiếp vào report.

In [ ]:
# Phase 10 - Cell 1 - Tạo bảng kết luận eval
phase10_eval_conclusion = pd.DataFrame([
    {'section': 'performance', 'conclusion': f'Final model test WAPE = {test_row["final_model_wape"]:.4f}, baseline test WAPE = {test_row["baseline_wape"]:.4f}.'},
    {'section': 'performance', 'conclusion': f'Model cải thiện WAPE tuyệt đối {test_row["wape_improvement_abs"]:.4f}, tương đương {test_row["wape_improvement_pct_vs_baseline"]:.2%} so với baseline.'},
    {'section': 'stability', 'conclusion': f'Validation WAPE = {validation_row["final_model_wape"]:.4f}, test WAPE = {test_row["final_model_wape"]:.4f}, gap = {wape_gap:.4f}.'},
    {'section': 'stability', 'conclusion': f'Prediction âm: {negative_prediction_count}; std ratio prediction/actual = {std_ratio_pred_vs_actual:.3f}.'},
    {'section': 'strength', 'conclusion': 'Model thắng yesterday baseline trên test và được đánh giá theo time-based split.'},
    {'section': 'strength', 'conclusion': 'Test set không dùng để tuning, nên kết quả test phù hợp vai trò đánh giá cuối cùng.'},
    {'section': 'weakness', 'conclusion': f'Model vẫn sai mạnh ở một số ngày biến động lớn; top error day lớn nhất có absolute error {phase6_top_error_days.iloc[0]["final_model_abs_error"]:,.0f}.'},
    {'section': 'weakness', 'conclusion': f'Nhóm doanh thu cần theo dõi nhất là {max_group}; tháng cần theo dõi nhất là {max_month}.'},
    {'section': 'recommendation', 'conclusion': 'Khi dùng thực tế, nên theo dõi WAPE theo tháng và theo nhóm doanh thu.'},
    {'section': 'recommendation', 'conclusion': 'Nên cập nhật hoặc kiểm tra lại model khi dữ liệu mới có dấu hiệu distribution shift.'},
])

phase10_eval_conclusion.to_csv(EVAL_TABLE_DIR / 'phase10_eval_conclusion.csv', index=False, encoding='utf-8-sig')
print(phase10_eval_conclusion.to_string(index=False))

In [ ]:
# Phase 10 - Cell 2 - Tạo summary ngắn cho report
phase10_eval_report_summary = pd.DataFrame([
    {'item': 'final_model', 'value': str(summary_map.get('final_model', 'lightgbm'))},
    {'item': 'baseline_model', 'value': BASELINE_COL},
    {'item': 'test_wape_final_model', 'value': test_row['final_model_wape']},
    {'item': 'test_wape_baseline', 'value': test_row['baseline_wape']},
    {'item': 'test_wape_improvement_pct_vs_baseline', 'value': test_row['wape_improvement_pct_vs_baseline']},
    {'item': 'test_mae_final_model', 'value': test_row['final_model_mae']},
    {'item': 'test_rmse_final_model', 'value': test_row['final_model_rmse']},
    {'item': 'validation_test_wape_gap', 'value': wape_gap},
    {'item': 'negative_prediction_count', 'value': negative_prediction_count},
    {'item': 'std_ratio_pred_vs_actual', 'value': std_ratio_pred_vs_actual},
    {'item': 'remaining_risk', 'value': f'Theo dõi {max_month} và nhóm {max_group} vì có WAPE cao nhất trong các nhóm đã phân tích.'},
])

phase10_eval_report_summary.to_csv(EVAL_TABLE_DIR / 'phase10_eval_report_summary.csv', index=False, encoding='utf-8-sig')
print(phase10_eval_report_summary.to_string(index=False))

## Kết luận Phase 10

- Final model thắng baseline trên test theo WAPE.
- Test WAPE gần validation WAPE, nên chưa thấy dấu hiệu performance tụt mạnh trên test.
- Model không tạo prediction âm, nhưng vẫn cần theo dõi ngày doanh thu biến động lớn.
- Kết luận eval đã được lưu thành bảng để đưa trực tiếp vào báo cáo.

## Phase 11 - Chuẩn bị bảng và biểu đồ báo cáo eval

### Mục tiêu

- Gom các bảng quan trọng thành output cuối dễ dùng trong slide/report.
- Copy các biểu đồ chính sang tên file report-ready.
- Tạo checklist output cuối.

In [ ]:
# Phase 11 - Cell 1 - Tạo bảng report-ready
eval_metric_summary = phase4_eval_metric_comparison_detail.copy()
eval_error_summary = pd.concat([
    phase6_top_error_days.assign(summary_type='top_error_day').head(10),
], ignore_index=True)
eval_risk_summary = phase8_eval_risk_summary.copy()
eval_feature_explanation = phase9_eval_feature_explanation.copy()
eval_final_conclusion = phase10_eval_conclusion.copy()

eval_metric_summary.to_csv(EVAL_TABLE_DIR / 'eval_metric_summary.csv', index=False, encoding='utf-8-sig')
eval_error_summary.to_csv(EVAL_TABLE_DIR / 'eval_error_summary.csv', index=False, encoding='utf-8-sig')
eval_risk_summary.to_csv(EVAL_TABLE_DIR / 'eval_risk_summary.csv', index=False, encoding='utf-8-sig')
eval_feature_explanation.to_csv(EVAL_TABLE_DIR / 'eval_feature_explanation.csv', index=False, encoding='utf-8-sig')
eval_final_conclusion.to_csv(EVAL_TABLE_DIR / 'eval_final_conclusion.csv', index=False, encoding='utf-8-sig')

print('eval_metric_summary')
print(eval_metric_summary.to_string(index=False))
print('\neval_final_conclusion')
print(eval_final_conclusion.to_string(index=False))

In [ ]:
# Phase 11 - Cell 2 - Copy biểu đồ chính sang tên report-ready
import shutil

figure_exports = {
    'phase5_actual_vs_predicted_test.png': 'eval_actual_vs_predicted.png',
    'phase6_error_timeline_test.png': 'eval_error_timeline.png',
    'phase4_wape_model_vs_baseline.png': 'eval_wape_model_vs_baseline.png',
    'phase9_top_feature_importance.png': 'eval_top_feature_importance.png',
}

copy_rows = []
for source_name, target_name in figure_exports.items():
    source_path = EVAL_FIG_DIR / source_name
    target_path = EVAL_FIG_DIR / target_name
    assert source_path.exists(), f'Thiếu hình nguồn: {source_path}'
    shutil.copyfile(source_path, target_path)
    copy_rows.append({'source': source_name, 'target': target_name, 'exists': target_path.exists()})

phase11_figure_export_check = pd.DataFrame(copy_rows)
print(phase11_figure_export_check.to_string(index=False))

In [ ]:
# Phase 11 - Cell 3 - Checklist output cuối
final_output_files = [
    EVAL_TABLE_DIR / 'eval_metric_summary.csv',
    EVAL_TABLE_DIR / 'eval_error_summary.csv',
    EVAL_TABLE_DIR / 'eval_risk_summary.csv',
    EVAL_TABLE_DIR / 'eval_feature_explanation.csv',
    EVAL_TABLE_DIR / 'eval_final_conclusion.csv',
    EVAL_FIG_DIR / 'eval_actual_vs_predicted.png',
    EVAL_FIG_DIR / 'eval_error_timeline.png',
    EVAL_FIG_DIR / 'eval_wape_model_vs_baseline.png',
    EVAL_FIG_DIR / 'eval_top_feature_importance.png',
]

phase11_eval_output_checklist = pd.DataFrame({
    'output_file': [path.name for path in final_output_files],
    'path': [str(path) for path in final_output_files],
    'exists': [path.exists() for path in final_output_files],
})

phase11_eval_output_checklist.to_csv(EVAL_TABLE_DIR / 'phase11_eval_output_checklist.csv', index=False, encoding='utf-8-sig')
assert phase11_eval_output_checklist['exists'].all(), 'Có output cuối bị thiếu.'
print(phase11_eval_output_checklist.to_string(index=False))

## Kết luận Phase 11

- Các bảng report-ready đã được lưu trong `eval_outputs/tables`.
- Các biểu đồ chính đã được copy sang tên file cuối trong `eval_outputs/figures`.
- Notebook eval hoàn thành đủ Phase 0-11 theo `eval_task.md`.
- Toàn bộ quy trình eval không train lại model, không tuning bằng test và không chọn lại model bằng test.